The following code was used to process the articles generated by the GDELT API

In [ ]:
!pip install newspaper3k
!pip install "lxml[html_clean]"
import requests
import re, re as regex
from bs4 import BeautifulSoup
import time
import pandas as pd
from newspaper import Article
import nltk
from datetime import datetime
import ast
import numpy as np
import dateutil.parser as dparser
# Used for exporting .csv file to the Drive
import shutil
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Import necessary packages to upload and clean scraper output
#Make sure to upload both scraper output and weekly counts
from google.colab import files
uploaded = files.upload()   # opens a file picker

Saving all_articles_en.csv to all_articles_en.csv
Saving weekly_alerts.csv to weekly_alerts.csv
Saving keywords.csv to keywords.csv


In [ ]:
#  Set global export location - Used for all exports within the file unless explicitly overwritten.
destination_folder = '/content/drive/MyDrive/Colab Notebooks/MAP6114 - Machine Learning/MAP 6114 Project/Model_Cheyanne'

In [ ]:
def save_and_export_to_drive(df, source_file_name, destination_folder_name):
  # Save file
  df.to_csv(source_file_name, index = False)

  # Export file to drive
  shutil.copy(source_file_name, destination_folder_name)

  # Returns file names in the destination folder as a Python SList
  # Format: ['file1.csv file2.csv'] - Files are not delimited by commas
  files = !ls "{destination_folder}"

  # Extract the file names and split on the space to format the file names as a list.
  all_filenames = files[0].split()
  print(all_filenames)

  # Verify that the file was copied to the destination folder.
  if source_file_name in all_filenames:
    print(f"'{source_file_name}' successfully copied to '{destination_folder}'")
  else:
    print(f"'{source_file_name}' not found in '{destination_folder}'")


In [ ]:
#final_df_list = pd.read_csv("final_df_list.csv")
all_articles = pd.read_csv("all_articles_en.csv")
weekly_alerts = pd.read_csv("weekly_alerts.csv")
keywords = pd.read_csv("keywords.csv")

In [ ]:
# Normalize keyword list
keywords_df = pd.read_csv("keywords.csv", header=None, names=['keywords', 'associated_country'])
keywords_df["keyword_norm"] = keywords_df["keywords"].str.lower()

# Create a mapping from normalized keyword to associated country
keyword_to_country_map = dict(zip(keywords_df["keyword_norm"], keywords_df["associated_country"]))

def assign_keywords_and_country(text):
    text_lower = str(text).lower()
    matched_keywords = []
    matched_countries = []
    for kw_norm in keywords_df["keyword_norm"]:
        if kw_norm in text_lower:
            matched_keywords.append(kw_norm)
            matched_countries.append(keyword_to_country_map[kw_norm])

    # Return matched keywords as a list, and matched countries as a list
    return matched_keywords, matched_countries

# Apply to article title column to get keywords and countries
all_articles["keywords"], all_articles["country_list"] = zip(*all_articles["title"].apply(assign_keywords_and_country))

# Join multiple countries into a single string if an article matches multiple country keywords
all_articles["country"] = all_articles["country_list"].apply(lambda x: ";".join(x) if x else None)

# Drop the temporary country_list column
all_articles = all_articles.drop(columns=['country_list'])

source_file = 'all_articles_en_tagged.csv'

save_and_export_to_drive(all_articles, source_file, destination_folder)

['all_articles_en.csv', 'all_articles_en_tagged.csv', 'Model_Cheyanne.ipynb']
'all_articles_en_tagged.csv' successfully copied to '/content/drive/MyDrive/Colab Notebooks/MAP6114 - Machine Learning/MAP 6114 Project/Model_Cheyanne'


In [ ]:
total_articles = len(all_articles)
print(f"Total number of articles: {total_articles}")

missing_urls = all_articles[all_articles["url"].isnull()]
print(f"Number of missing URLs: {len(missing_urls)}")

missing_week_start = all_articles[all_articles["week_start"].isnull()]
print(f"Number of missing week_start: {len(missing_week_start)}")

missing_week_end = all_articles[all_articles["week_end"].isnull()]
print(f"Number of missing week_end: {len(missing_week_end)}")

# Create 100 partitions.
partitions = np.array_split(all_articles, 100)

print(f"Number of partitions: {len(partitions)}")
print(f"Number of rows in the first and last partitions, respectively: {len(partitions[0]), len(partitions[99])}")

Total number of articles: 56055
Number of missing URLs: 0
Number of missing week_start: 0
Number of missing week_end: 0
Number of partitions: 100
Number of rows in the first and last partitions, respectively: (561, 560)


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [ ]:
today = datetime.today().strftime('%Y-%m-%d')
date_keywords = ['editeddate','releaseddate','dateCreated','datePublished','dateModified','lastPublishedDate']

In [ ]:
final_df_list = []
error_df_list = []

In [ ]:
def append_error(error_df_list, url, country, error_message, title=None, keywords=None, meta=None, text=None, summary=None, date=None):
    """
    Append an error row to the error_df_list.
    """
    error_df = pd.DataFrame({'url':[url]})
    error_df['error message'] = error_message
    error_df['country'] = country
    error_df['title'] = title
    error_df['keywords'] = keywords
    error_df['meta'] = meta
    error_df['text'] = text
    error_df['summary'] = summary
    error_df['date'] = date
    error_df_list.append(error_df)

In [ ]:
def batch_process_articles(partitions, partition_start, partition_end, final_df_list, error_df_list, date_keys=date_keywords, today_date=today, destination=destination_folder):
    function_start_time = time.time()

    for partition_num in range(partition_start, partition_end):
        partition = partitions[partition_num]
        error_num = 0
        print(f"Processing partition {partition_num}")
        print(f"Total number of errors: {error_num}")
        partition_start_time = time.time()

        for i, row in partition.iterrows():
            if i % 20 == 0:
                if partition_num < 100:
                  partition_row_num = i - ((i // 561) * 561)
                else:
                  partition_row_num = i - ((i // 560) * 560)
                print(f"Processing row {partition_row_num} in partition {partition_num}")
                print(f"Total number of errors: {error_num}")


            url = row['url']

            try:
                toi_article = Article(url, language='en')
                toi_article.download()
                toi_article.parse()
                toi_article.nlp()

                article_title = str(toi_article.title)
                article_text = str(toi_article.text).replace('\n', '')
                meta_data_dict = toi_article.meta_data

                # --- DATE HANDLING ---
                date_val = None

                # 1) publish_date
                if toi_article.publish_date is not None:
                    try:
                        date_val = pd.to_datetime(toi_article.publish_date)
                    except Exception:
                        date_val = None

                # 2) date-like keys in meta dict
                if date_val is None and isinstance(meta_data_dict, dict):
                    for key in date_keys:
                        if key in meta_data_dict:
                            try:
                                date_val = pd.to_datetime(meta_data_dict[key])
                                break
                            except Exception:
                                pass

                # 3) regex fallback
                if date_val is None:
                    meta_str = str(meta_data_dict)
                    for key in date_keys:
                        m = re.search(rf"{key}[^0-9]*(\d{{4}}[-/]\d{{2}}[-/]\d{{2}})", meta_str)
                        if m:
                            try:
                                date_val = pd.to_datetime(m.group(1))
                                break
                            except Exception:
                                pass

                # 4) last resort: today
                if date_val is None:
                    date_val = pd.to_datetime(today_date)

                # --- SOFT ERROR: title == 'MSN', 'STUFF', 'POLITICS', 'ZEROHEDGE' ---
                article_title_formatted = article_title.strip().upper()
                if article_title_formatted == "MSN" or article_title_formatted == "STUFF" or article_title_formatted == "POLITICS" or article_title_formatted == 'ZEROHEDGE':
                    error_num += 1
                    append_error(
                        error_df_list,
                        url,
                        row['country'],
                        error_message="Title returned as 'MSN'",
                        title=article_title,
                        date=date_val,
                        keywords=toi_article.keywords,
                        meta=str(meta_data_dict),
                        text=article_text,
                        summary=toi_article.summary
                    )
                    continue

                # --- FINAL DATAFRAME ---
                df_temp_final = pd.DataFrame({
                    'title': [article_title],
                    'meta': [str(meta_data_dict)],
                    'text': [article_text[:5000] if len(article_text) > 5000 else article_text],
                    'summary': [str(toi_article.summary)],
                    'date': [date_val],
                    'keywords': [toi_article.keywords],
                    'url': [url],
                    'country': [row['country']]
                })

                final_df_list.append(df_temp_final)

            except Exception as e:
                error_num += 1
                append_error(
                    error_df_list,
                    url,
                    row['country'],
                    error_message=str(e)
                )
                continue

        partition_end_time = time.time()
        print(f"Completed partition {partition_num} in {partition_end_time - partition_start_time:.2f} seconds. Total elapsed time: {partition_end_time - function_start_time:.2f} seconds")
        print(f"Number of errors: {error_num}")

        # --- SAVE INTERIM DATA ---
        if final_df_list:
            articles_raw_interim = pd.concat(final_df_list, ignore_index=True)
            print(f"Total number of articles: {len(articles_raw_interim)}")
            source_file = f'articles_raw_{partition_num}.csv'
            source_file = f'articles_raw_{partition_num}.csv' #twice?
            save_and_export_to_drive(articles_raw_interim, source_file, destination)
        else:
            articles_raw_interim = pd.DataFrame()
            print(f'Final DF List is empty. Please evaluate the function call.')

        if error_df_list:
            error_df_interim = pd.concat(error_df_list, ignore_index=True)
            print(f"Total number of errors: {len(error_df_interim)}")
            source_file = f'error_df_list_{partition_num}.csv'
            source_file = f'error_df_list_{partition_num}.csv'
            save_and_export_to_drive(error_df_interim, source_file, destination)
        else:
            error_df_interim = pd.DataFrame()
            print(f'There are no errors within the partition.')

    return articles_raw_interim, error_df_interim
